# BKK Transit Analytics — Delay Prediction

End-to-end analysis of Budapest public transit delays: exploratory analysis, feature engineering, and LSTM-based prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Load data — auto-detects Colab vs local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/bkk-transit-analytics/analytics_master.csv'
except ImportError:
    DATA_PATH = '../data/analytics_master.csv'

# Only load columns we actually use
use_cols = [
    'delay_seconds', 'scheduled_time', 'actual_time', 'date_iso',
    'year', 'month', 'day', 'hour', 'minute', 'weekday', 'is_weekend', 'part_of_day',
    'temp', 'rain', 'wind_speed', 'humidity', 'cloudiness', 'weather_category',
    'route_id', 'route_name', 'route_type',
    'stop_id', 'vehicle_natural_id',
]

df = pd.read_csv(
    DATA_PATH,
    usecols=use_cols,
    parse_dates=['scheduled_time', 'actual_time', 'date_iso']
)

# Downcast to save memory: float64 -> float32, int64 -> int32
for col in df.select_dtypes('float64').columns:
    df[col] = df[col].astype(np.float32)
for col in df.select_dtypes('int64').columns:
    df[col] = df[col].astype(np.int32)

print(f'Loaded {len(df):,} rows')
print(f'Date range: {df["date_iso"].min().date()} to {df["date_iso"].max().date()}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

---
# Part 1: Exploratory Data Analysis

## 1.1 Dataset Overview

In [ ]:
print(f'Shape: {df.shape}')
print(f'Unique routes: {df["route_id"].nunique()}')
print(f'Unique stops:  {df["stop_id"].nunique()}')
print(f'Unique vehicles: {df["vehicle_natural_id"].nunique()}')
print()
print('--- Missing values ---')
missing = df.isnull().sum()
print(missing[missing > 0])
print()
df.describe()

## 1.2 Target Variable: `delay_seconds`

In [ ]:
delay = df['delay_seconds'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(delay.clip(-1800, 1800), bins=80, edgecolor='white', alpha=0.7)
axes[0].set_title('Delay Distribution (clipped to +/-30 min)')
axes[0].set_xlabel('Delay (seconds)')
axes[0].set_ylabel('Count')

axes[1].boxplot(delay, vert=True)
axes[1].set_title('Delay Box Plot')
axes[1].set_ylabel('Delay (seconds)')

plt.tight_layout()
plt.show()

print(f'Mean:   {delay.mean():.1f}s')
print(f'Median: {delay.median():.1f}s')
print(f'Std:    {delay.std():.1f}s')
print(f'Min:    {delay.min():.0f}s  |  Max: {delay.max():.0f}s')

## 1.3 Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Average delay by hour
df.groupby('hour')['delay_seconds'].mean().plot(ax=axes[0, 0], marker='o')
axes[0, 0].set_title('Average Delay by Hour')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Delay (s)')

# Average delay by weekday
weekday_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
by_weekday = df.groupby('weekday')['delay_seconds'].mean()
axes[0, 1].bar(by_weekday.index, by_weekday.values)
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(weekday_labels)
axes[0, 1].set_title('Average Delay by Weekday')
axes[0, 1].set_ylabel('Delay (s)')

# Average delay by part of day
pod_order = ['Morning', 'Afternoon', 'Evening', 'Night']
by_pod = df.groupby('part_of_day')['delay_seconds'].mean().reindex(pod_order)
axes[1, 0].bar(by_pod.index, by_pod.values, color=['#f0ad4e', '#5bc0de', '#d9534f', '#337ab7'])
axes[1, 0].set_title('Average Delay by Part of Day')
axes[1, 0].set_ylabel('Delay (s)')

# Heatmap: hour x weekday
pivot = df.pivot_table(values='delay_seconds', index='hour', columns='weekday', aggfunc='mean')
pivot.columns = weekday_labels
sns.heatmap(pivot, cmap='RdYlGn_r', ax=axes[1, 1], fmt='.0f', annot=True, annot_kws={'size': 7})
axes[1, 1].set_title('Mean Delay: Hour x Weekday')

plt.tight_layout()
plt.show()

## 1.4 Weather Impact

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Delay by weather category
weather_avg = df.groupby('weather_category')['delay_seconds'].mean().sort_values()
axes[0, 0].barh(weather_avg.index, weather_avg.values)
axes[0, 0].set_title('Average Delay by Weather')
axes[0, 0].set_xlabel('Delay (s)')

# Scatter: delay vs temperature
sample = df.sample(min(5000, len(df)), random_state=42)
axes[0, 1].scatter(sample['temp'], sample['delay_seconds'], alpha=0.1, s=5)
axes[0, 1].set_title('Delay vs Temperature')
axes[0, 1].set_xlabel('Temperature (C)')
axes[0, 1].set_ylabel('Delay (s)')

# Scatter: delay vs rain
axes[1, 0].scatter(sample['rain'], sample['delay_seconds'], alpha=0.1, s=5)
axes[1, 0].set_title('Delay vs Rain')
axes[1, 0].set_xlabel('Rain (mm)')
axes[1, 0].set_ylabel('Delay (s)')

# Correlation matrix
num_cols = ['delay_seconds', 'temp', 'rain', 'wind_speed', 'humidity', 'cloudiness', 'hour', 'weekday']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

## 1.5 Route Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delay by route type
rt_delay = df.groupby('route_type')['delay_seconds'].mean().sort_values(ascending=False)
axes[0].bar(rt_delay.index, rt_delay.values)
axes[0].set_title('Average Delay by Route Type')
axes[0].set_ylabel('Delay (s)')
axes[0].tick_params(axis='x', rotation=45)

# Top 10 most delayed routes
top10 = df.groupby(['route_id', 'route_name'])['delay_seconds'].mean().nlargest(10)
labels = [f'{name} ({rid})' for (rid, name) in top10.index]
axes[1].barh(labels, top10.values)
axes[1].set_title('Top 10 Most Delayed Routes')
axes[1].set_xlabel('Mean Delay (s)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('Events per route type:')
print(df['route_type'].value_counts())

---
# Part 2: Feature Engineering

First, filter to the top 50 routes by event count. Routes with few observations produce very few training sequences and waste memory. This keeps the most representative data.

## 2.1 Filter & Clean Data

Keep top 50 routes by event count, drop rows without delay, clip outliers to +/-30 minutes.

In [ ]:
# Keep top 50 routes by event count
top_routes = df['route_id'].value_counts().head(50).index
df = df[df['route_id'].isin(top_routes)].copy()
print(f'Kept top 50 routes: {len(df):,} rows ({len(top_routes)} routes)')

# Drop rows without delay, clip outliers
df = df.dropna(subset=['delay_seconds'])
df['delay_seconds'] = df['delay_seconds'].clip(-1800, 1800)

gc.collect()
print(f'After cleaning: {len(df):,} rows')
print(f'Delay range: [{df["delay_seconds"].min():.0f}, {df["delay_seconds"].max():.0f}]')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 2.2 Create Features

- **Trigonometric time encoding**: hour, minute, weekday, month encoded as sin/cos pairs.
- **Weather**: numerical weather columns + one-hot weather category.
- **Route type**: one-hot encoded.
- **Binary flags**: weekend, rush hour.
- **Interaction**: rain during rush hour.

In [ ]:
# Trigonometric time encoding
for col, period in [('hour', 24), ('minute', 60), ('weekday', 7), ('month', 12)]:
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)

# Binary flags
df['is_weekend'] = df['is_weekend'].astype(int)
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 16, 17, 18]).astype(int)

# Interaction: rain during rush hour
df['rain'] = df['rain'].fillna(0)
df['rain_x_rush'] = df['rain'] * df['is_rush_hour']

# One-hot encode route_type and weather_category
rt_dummies = pd.get_dummies(df['route_type'], prefix='rt').astype(int)
wc_dummies = pd.get_dummies(df['weather_category'], prefix='wc').astype(int)
df = pd.concat([df, rt_dummies, wc_dummies], axis=1)

print(f'Route type columns: {list(rt_dummies.columns)}')
print(f'Weather columns: {list(wc_dummies.columns)}')

## 2.3 Define Feature Columns

In [ ]:
trig_cols = [f'{c}_{fn}' for c in ['hour', 'minute', 'weekday', 'month'] for fn in ['sin', 'cos']]
weather_cols = ['temp', 'rain', 'wind_speed', 'humidity', 'cloudiness']
binary_cols = ['is_weekend', 'is_rush_hour']
interaction_cols = ['rain_x_rush']
rt_cols = [c for c in df.columns if c.startswith('rt_')]
wc_cols = [c for c in df.columns if c.startswith('wc_')]

feature_cols = trig_cols + weather_cols + binary_cols + interaction_cols + rt_cols + wc_cols
target_col = 'delay_seconds'

print(f'Total features: {len(feature_cols)}')
for group, cols in [('Trig time', trig_cols), ('Weather', weather_cols),
                     ('Binary', binary_cols), ('Interaction', interaction_cols),
                     ('Route type', rt_cols), ('Weather cat', wc_cols)]:
    print(f'  {group}: {len(cols)} — {cols}')

keep_cols = feature_cols + [target_col, 'route_id', 'scheduled_time', 'date_iso']
df = df[keep_cols].copy()
gc.collect()
print(f'\nKept {len(keep_cols)} columns')

---
# Part 3: Scaling & Train/Validation/Test Split

The data is split **temporally** (by date, not randomly)

In [ ]:
# Temporal split: 80% train, 10% validation, 10% test
dates = sorted(df['date_iso'].dt.date.unique())
n = len(dates)
train_end = dates[int(n * 0.8)]
val_end = dates[int(n * 0.9)]

train = df[df['date_iso'].dt.date <= train_end].copy()
val = df[(df['date_iso'].dt.date > train_end) & (df['date_iso'].dt.date <= val_end)].copy()
test = df[df['date_iso'].dt.date > val_end].copy()

# Free the full dataframe — we only need the splits from here
del df
gc.collect()

print(f'Train: {len(train):,} rows  (up to {train_end})')
print(f'Val:   {len(val):,} rows  ({train_end} to {val_end})')
print(f'Test:  {len(test):,} rows  (after {val_end})')

## 3.1 Scale Features

StandardScaler is fitted on the training set only, then applied to validation and test sets.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fill remaining NaN in features
for split in [train, val, test]:
    split[feature_cols] = split[feature_cols].fillna(0)

# Scale continuous features (not binary/one-hot)
scale_cols = trig_cols + weather_cols + interaction_cols

scaler = StandardScaler()
train[scale_cols] = scaler.fit_transform(train[scale_cols])
val[scale_cols] = scaler.transform(val[scale_cols])
test[scale_cols] = scaler.transform(test[scale_cols])

print(f'Scaled {len(scale_cols)} continuous columns')
print(f'Example (train mean after scaling): {train[scale_cols].mean().abs().mean():.4f} (should be ~0)')

## 3.2 Create Sequences for LSTM

LSTM expects sequences of consecutive observations. For each route, sliding windows of 12 timesteps are created.

In [ ]:
SEQ_LENGTH = 12

def create_sequences(data, seq_length=SEQ_LENGTH):
    """Create sequences grouped by route, sorted by time."""
    X, y = [], []
    for _, group in data.groupby('route_id'):
        group = group.sort_values('scheduled_time')
        features = group[feature_cols].values
        target = group[target_col].values
        for i in range(len(features) - seq_length):
            X.append(features[i : i + seq_length])
            y.append(target[i + seq_length])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train, y_train = create_sequences(train)
del train; gc.collect()

X_val, y_val = create_sequences(val)
del val; gc.collect()

X_test, y_test = create_sequences(test)
del test; gc.collect()

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}    y_test:  {y_test.shape}')
print(f'Total sequence memory: {(X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e6:.1f} MB')

---
# Part 4: LSTM Model

Architecture: LSTM(32) -> Dropout -> BatchNorm -> LSTM(16) -> Dropout -> Dense(16) -> Dense(1)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')

model = Sequential([
    LSTM(32, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    BatchNormalization(),
    LSTM(16, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

## 4.1 Train

In [ ]:
callbacks = [
    EarlyStopping(patience=2, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=1, factor=0.5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

## 4.2 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation', linestyle='--')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation', linestyle='--')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# Part 5: Evaluation

Compare the LSTM model against a simple baseline (predict the training set mean for every sample).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# LSTM predictions
y_pred = model.predict(X_test, verbose=0).flatten()

# Baseline: always predict the training mean
y_baseline = np.full_like(y_test, y_train.mean())

# Results table
results = pd.DataFrame([
    {
        'Model': 'Mean Baseline',
        'MAE': mean_absolute_error(y_test, y_baseline),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_baseline)),
        'R2': r2_score(y_test, y_baseline),
    },
    {
        'Model': 'LSTM',
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred),
    },
])

print(results.to_string(index=False))

## 5.1 Prediction Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.1, s=5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_title('Predicted vs Actual')
axes[0].set_xlabel('Actual Delay (s)')
axes[0].set_ylabel('Predicted Delay (s)')

# Residuals
axes[1].scatter(y_pred, residuals, alpha=0.1, s=5)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_title('Residuals')
axes[1].set_xlabel('Predicted Delay (s)')
axes[1].set_ylabel('Residual (s)')

# Error distribution
axes[2].hist(residuals, bins=60, edgecolor='white', alpha=0.7)
axes[2].set_title('Error Distribution')
axes[2].set_xlabel('Error (s)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Mean error:  {residuals.mean():.1f}s')
print(f'Std error:   {residuals.std():.1f}s')